# 2 · Random Forest — Predicting Flight Delays

A Random Forest is an **ensemble of decision trees**. Each tree is trained on a bootstrap sample of the rows and a random subset of the features; the forest averages their votes. This reduces variance without adding bias, which is why Random Forest is often a strong no-tuning baseline.

**What we'll do**
1. Load the cleaned data and encode categoricals
2. Chronological train/test split
3. Fit `RandomForestClassifier` for the `is_delayed` target
4. Evaluate with accuracy / precision / recall / F1 / ROC AUC / PR AUC
5. Inspect the confusion matrix, ROC, and PR curves
6. Look at feature importances
7. Bonus: regress the delay magnitude

## Setup

**Required packages**
```
pip install pandas numpy matplotlib seaborn scikit-learn xgboost
```
The dataset is `bwi_model_ready.csv` — already cleaned by `preprocess.py`.
It covers 14 days at Baltimore/Washington International (BWI), joined with hourly weather.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 5)

CATEGORICAL = ["carrier", "origin_airport", "destination_airport", "direction"]

df = pd.read_csv("bwi_model_ready.csv")
for col in CATEGORICAL:
    df[col] = df[col].astype("category")

print("shape:", df.shape)
df.head()


### Encode categoricals for Random Forest
sklearn's `RandomForestClassifier` does **not** accept string/category features natively. We one-hot encode the four categorical columns. Tree splits on one-hot columns are equivalent to "is this category present?" tests — simple and fast.

In [ ]:
df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)
print(f"columns after one-hot: {df_enc.shape[1]}")
df = df_enc  # keep using the same variable name downstream

### Train/test split (chronological)

In [ ]:
# Target: binary "is the flight delayed >= 15 min?"
# We drop rows where the label is missing (cancelled flights with no arrival time).
data = df.dropna(subset=["is_delayed"]).copy()
data["is_delayed"] = data["is_delayed"].astype(int)

TARGET = "is_delayed"
DROP_FROM_X = ["is_cancelled", "arrival_delay_minutes", "is_delayed"]
feature_cols = [c for c in data.columns if c not in DROP_FROM_X]

# Time-based split: BTS rows come pre-sorted by date.
# Using a chronological holdout is the right call for this kind of data —
# a random split would leak "future" information into the training set.
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
X_train, y_train = train[feature_cols], train[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

print(f"train rows: {len(X_train):,}   positive rate: {y_train.mean():.1%}")
print(f"test  rows: {len(X_test):,}   positive rate: {y_test.mean():.1%}")


### Fit the forest
Starting hyperparameters chosen for teaching, not for a competition:
- `n_estimators=200` — enough trees for stable estimates
- `max_depth=None` — let trees grow, rely on bagging to control variance
- `min_samples_leaf=5` — a tiny bit of regularization
- `class_weight="balanced"` — up-weight the minority (delayed) class

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print(f"trained forest with {model.n_estimators} trees, "
      f"avg depth ≈ {np.mean([t.get_depth() for t in model.estimators_]):.1f}")

### Metrics
Always report more than one metric. **Accuracy** is misleading on imbalanced data; **ROC AUC** summarizes ranking quality; **PR AUC** is the right summary when the positive class is rare.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

metrics = {
    "accuracy":  accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall":    recall_score(y_test, pred, zero_division=0),
    "f1":        f1_score(y_test, pred, zero_division=0),
    "roc_auc":   roc_auc_score(y_test, proba),
    "pr_auc":    average_precision_score(y_test, proba),
}
pd.Series(metrics).round(4).to_frame("value")


In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["on-time", "delayed"])
ax.set_yticklabels(["on-time", "delayed"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix")
plt.tight_layout(); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
prec, rec, _ = precision_recall_curve(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
axes[0].legend()

axes[1].plot(rec, prec, label=f"AP = {metrics['pr_auc']:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray",
                label=f"baseline = {y_test.mean():.2f}")
axes[1].set(title="Precision–Recall curve", xlabel="Recall", ylabel="Precision")
axes[1].legend()
plt.tight_layout(); plt.show()


### Feature importance
`feature_importances_` returns **mean decrease in impurity** — how much each feature reduces Gini on average across all splits. Good for a quick look, but biased toward high-cardinality features. Permutation importance is a more honest alternative (commented out below because it's slower).

In [ ]:
imp = (pd.Series(model.feature_importances_, index=feature_cols)
         .sort_values(ascending=False)
         .head(20))
ax = imp[::-1].plot.barh(color='forestgreen')
ax.set(title='Top 20 feature importances (Gini)', xlabel='importance')
plt.tight_layout(); plt.show()

In [ ]:
# Optional — more honest but slower:
# from sklearn.inspection import permutation_importance
# perm = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=0, n_jobs=-1)
# pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False).head(15)

### Bonus: regressing the delay magnitude
Classification tells us *whether* a flight is delayed. Regression on the minutes themselves tells us *how badly*. Train only on non-cancelled rows with a known arrival time.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

reg_data = df.dropna(subset=['arrival_delay_minutes']).copy()
reg_train = reg_data.iloc[:int(len(reg_data) * 0.8)]
reg_test  = reg_data.iloc[int(len(reg_data) * 0.8):]
y_reg_train = reg_train['arrival_delay_minutes']
y_reg_test  = reg_test['arrival_delay_minutes']
X_reg_train = reg_train[feature_cols]
X_reg_test  = reg_test[feature_cols]

reg = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                            random_state=42, n_jobs=-1)
reg.fit(X_reg_train, y_reg_train)
pred_reg = reg.predict(X_reg_test)
print(f"MAE:  {mean_absolute_error(y_reg_test, pred_reg):.2f} min")
print(f"R²:   {r2_score(y_reg_test, pred_reg):.3f}")

### Writeup prompts
- Compare your confusion matrix to the PR curve — are you catching most delays or being too cautious?
- Does the top feature importance match what you expected from the data-exploration notebook?
- If a user wanted **high recall** (catch every delay) what threshold would you choose instead of 0.5?